#### This first part imports the pandas library and reads in the CSV file.

In [ ]:
import requests
from io import BytesIO
study_id = " "
response = requests.post( 
    "https://studies.beiwe.org/get-participant-table-data/v1",
        data={
            "access_key": " ",
            "secret_key": " ",
            "data_format": "csv",
            "study_id": study_id,
    },
    allow_redirects=False,
)
if response.status_code == 200:
    participants_table = f"participant_table_{study_id}.csv"
    print(f"The data has been written to {participants_table}.")
    with open(participants_table, "wb") as f:
        f.write(response.content)
else:
    print(response.content)
    print(f"There was an error downloading the data; The request returned an HTTP error of {response.status_code}")




200
b'Created On,Patient ID,Status,OS Type,Baseline,Virtual Visit #2,First Registration Date,Last Registration,Last Timezone,Last Upload,Last Survey Download,Last Set Password,Last Push Token Update,Last Device Settings Update,Last OS Version,App Version Code,App Version Name,Last Heartbeat\r\n2026-01-06,3p4eblir,Active (just now),ANDROID,2026-01-06,,2026-01-06 12:13:38 (EST),2026-01-06 12:13:38 (EST),America/Chicago,2026-01-14 21:11:09 (EST),2026-01-14 21:09:07 (EST),None,2026-01-14 21:20:08 (EST),None,16,84,3.6.0,2026-01-14 21:37:44 (EST)\r\n2025-12-04,443yroib,Active (just now),IOS,2025-12-04,,2025-12-04 10:45:24 (EST),2025-12-04 10:45:24 (EST),America/New_York,2026-01-14 21:35:17 (EST),2026-01-14 21:34:32 (EST),None,2026-01-09 04:19:40 (EST),2026-01-14 21:17:51 (EST),18.7.2,2.5.6,2025.11,2026-01-14 21:37:51 (EST)\r\n2025-04-08,555gcdo6,Inactive,IOS,2025-04-08,,2025-04-08 11:01:31 (EDT),2025-04-08 11:01:31 (EDT),America/New_York,2025-06-18 15:05:55 (EDT),2025-06-18 15:11:30 (EDT),No

In [ ]:
import pandas as pd
with open(participants_table, "rb") as f:
    df = pd.read_csv(f)
    # this will not work unless the participants_table variable in cell above has been populated and the file exists

#### First, we will get the number of participants who are registered. (that is, they have a First Registration Date that is not "None")

In [62]:
registered_count = df["First Registration Date"].notna().sum()
print(f"Participants registered: {registered_count}")

Participants registered: 40


#### Next, we will get the number of participants who are CURRENTLY registered. (anything BUT "Permanantly Retired" or "Not Registered")

In [61]:
currentlyreg_count = ((df["Status"] != "Permanently Retired") & (df["Status"] != "Not Registered")).sum()
print(f"Participants CURRENTLY registered: {currentlyreg_count}")

Participants CURRENTLY registered: 12


#### Now we will get counts of mobile device type (iOS and Android).

In [54]:
android_count = (df["OS Type"] == "ANDROID").sum()
ios_count = (df["OS Type"] == "IOS").sum()
print(f"iOS count: {ios_count} --- Android count: {android_count}")

iOS count: 27 --- Android count: 13


#### Finally we will get a breakdown of status for each type besides "Permanently Retired" or "Not Registered", ie. how many are "Active (Just Now)", etc.

In [58]:
status_counts = (
    df.loc[
        ~df["Status"].isin(["Permanently Retired", "Not Registered"]),
        "Status"
    ]
    .value_counts()
)

print(
    " --- ".join(
        f"{status}: {count}"
        for status, count in status_counts.items()
    )
)


Inactive: 8 --- Active (just now): 4
